# Machine Learning Practical Report
## K-Nearest Neighbors on Diabetes Data and K-Means / Hierarchical Clustering on Sales Data

**Subject:** Machine Learning / Data Science Practical  
**Student Name:** ______________________________  
**Class / Roll No.:** ___________________________  
**Date:** ______________________________________

---

## How to use this JupyterLab report

1. Keep `diabetes.csv` and `sales_data_sample.csv` in the **same folder** as this notebook.
2. Open this notebook in JupyterLab and select **Kernel → Restart Kernel and Run All Cells**.
3. Read the theory before running each practical. Run the cells from top to bottom.
4. In the K-Means section, inspect the Elbow graph and update `selected_k` to the elbow value you observe.
5. In the Hierarchical section, inspect the dendrogram and update `hierarchical_k` if it suggests a different number of clusters.

> **Important:** The CSV files were not attached when this notebook was prepared. The notebook checks for them and skips dataset-dependent calculations when they are absent. Put the actual CSV files beside the notebook and run all cells to generate your own confusion matrix, metrics, graphs, and cluster results. No example metric values are presented as if they came from your data.

### Learning objectives

After completing this report, you should be able to:

- Explain supervised and unsupervised machine learning.
- Prepare a dataset and separate input features from the target.
- Train a KNN classifier and evaluate it using a confusion matrix, accuracy, error rate, precision, and recall.
- Scale features before distance-based algorithms.
- Apply K-Means clustering and use the Elbow Method to select a reasonable K.
- Apply Agglomerative Hierarchical Clustering and interpret a dendrogram.

## Practical 1 — K-Nearest Neighbors (KNN) on `diabetes.csv`

### Aim

To implement the K-Nearest Neighbors classification algorithm on the diabetes dataset and calculate the confusion matrix, accuracy, error rate, precision, and recall.

### Theory: What is KNN?

K-Nearest Neighbors (KNN) is a **supervised machine learning algorithm**. It is commonly used for classification and can also be used for regression. In classification, KNN predicts the class of a new record by looking at the classes of its K closest records in the training dataset.

For this practical, the common Pima diabetes dataset uses the `Outcome` column as the target:

- `0` — the record is labelled as no diabetes.
- `1` — the record is labelled as diabetes.

The other columns, such as glucose, blood pressure, BMI, insulin, age, and pregnancies, are input features. This is a dataset exercise; a model prediction is **not** a medical diagnosis.

### Working of KNN

1. Choose the number of neighbours, K.
2. Calculate the distance from the new record to the training records.
3. Find the K records with the smallest distances.
4. Count the class labels among those neighbours.
5. Assign the majority class to the new record.

A common distance measure is Euclidean distance:

\[
 d(x,y)=\sqrt{\sum_{i=1}^{n}(x_i-y_i)^2}
\]

**Example:** If K = 5 and three of the five nearest neighbours belong to class 1 while two belong to class 0, KNN predicts class 1.

### Important terms

| Term | Meaning |
|---|---|
| Feature (X) | An input column used to make a prediction. |
| Target (y) | The column the model is trying to predict. |
| Training data | Data used to fit or train the model. |
| Testing data | Separate data used to evaluate predictions. |
| K | Number of nearest neighbours considered. |
| Feature scaling | Converting features to comparable scales so large-valued features do not dominate distance. |

### Why do we use StandardScaler?

KNN is based on distances. A feature measured in thousands can dominate a feature measured in tens. `StandardScaler` transforms each feature approximately to mean 0 and standard deviation 1, using statistics learned from the training data. This gives the features a comparable scale.

### Confusion matrix and performance measures

For binary classification, the confusion matrix compares actual labels with predicted labels:

| | Predicted 0 | Predicted 1 |
|---|---:|---:|
| **Actual 0** | True Negative (TN) | False Positive (FP) |
| **Actual 1** | False Negative (FN) | True Positive (TP) |

- **True Positive (TP):** Actual class is 1 and predicted class is 1.
- **True Negative (TN):** Actual class is 0 and predicted class is 0.
- **False Positive (FP):** Actual class is 0 but predicted class is 1.
- **False Negative (FN):** Actual class is 1 but predicted class is 0.

The formulas are:

\[
\text{Accuracy}=\frac{TP+TN}{TP+TN+FP+FN}
\]

\[
\text{Error Rate}=1-\text{Accuracy}
\]

\[
\text{Precision}=\frac{TP}{TP+FP}
\qquad
\text{Recall}=\frac{TP}{TP+FN}
\]

**Interpretation:** Accuracy is the fraction of all predictions that are correct. Precision answers, “Of the records predicted positive, how many were actually positive?” Recall answers, “Of all actual positive records, how many did the model identify?”

### Algorithm / procedure

1. Import the required Python libraries.
2. Read `diabetes.csv` using pandas.
3. Inspect the first records, dimensions, and missing values.
4. Separate features (`X`) and target (`y`).
5. In the common Pima dataset, treat zero readings in selected medical measurement columns as missing values where zero is not a realistic measurement.
6. Split the data into training and testing sets (80% / 20%).
7. Impute missing feature values using the training-set median and scale the features.
8. Train KNN with `K = 5`.
9. Predict the test-set target values.
10. Calculate and interpret the confusion matrix and performance measures.

### 1.1 Import the required libraries

In [1]:
# If any package is missing, run this in a JupyterLab cell:
# %pip install pandas numpy matplotlib scikit-learn scipy

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
    accuracy_score,
    precision_score,
    recall_score,
    classification_report
)

print("Libraries imported successfully.")

Libraries imported successfully.


### 1.2 Load and inspect the diabetes dataset

In [2]:
DIABETES_PATH = Path("diabetes.csv")
diabetes_available = DIABETES_PATH.exists()

if diabetes_available:
    df_diabetes = pd.read_csv(DIABETES_PATH)
    print("Dataset loaded successfully.")
    print("Shape (rows, columns):", df_diabetes.shape)
    display(df_diabetes.head())
    print("\nColumn names:")
    print(df_diabetes.columns.tolist())
    print("\nMissing values in each column:")
    display(df_diabetes.isna().sum().to_frame("Missing values"))
else:
    print("File not found: diabetes.csv")
    print("Place diabetes.csv in the same folder as this notebook, then rerun this cell and the cells below.")

File not found: diabetes.csv
Place diabetes.csv in the same folder as this notebook, then rerun this cell and the cells below.


**What does the inspection tell us?**

- `head()` displays the first five records.
- `shape` returns the number of rows and columns.
- `columns` lists the column names.
- `isna().sum()` counts missing values in each column.

The code below expects the commonly used target column `Outcome`. It finds the column without depending on its capitalization or surrounding spaces.

### 1.3 Separate input features (X) and target (y)

In [3]:
if diabetes_available:
    # Find the target column even if its capitalization differs.
    target_matches = [
        col for col in df_diabetes.columns
        if str(col).strip().lower() == "outcome"
    ]

    if not target_matches:
        raise ValueError(
            "The target column 'Outcome' was not found. "
            "Check the dataset columns and update target_matches if your target has another name."
        )

    target_column = target_matches[0]

    # Convert the target to numeric and remove rows with a missing target.
    target_values = pd.to_numeric(df_diabetes[target_column], errors="coerce")
    valid_target_rows = target_values.notna()

    X = df_diabetes.loc[valid_target_rows].drop(columns=[target_column]).copy()
    y = target_values.loc[valid_target_rows].astype(int)

    # KNN needs numeric input features. Non-numeric values become missing values.
    X = X.apply(pd.to_numeric, errors="coerce")

    # Remove any feature column that has no usable numeric values at all.
    X = X.dropna(axis=1, how="all")

    # Confirm the expected binary target labels.
    if not set(y.unique()).issubset({0, 1}):
        raise ValueError(f"Expected binary target labels 0 and 1, but found: {sorted(y.unique())}")

    print("Target column:", target_column)
    print("Feature matrix X shape:", X.shape)
    print("Target vector y shape:", y.shape)
    print("Target class counts:")
    display(y.value_counts().sort_index().rename(index={0: "No diabetes (0)", 1: "Diabetes (1)"}).to_frame("Records"))
else:
    print("Skipped: diabetes.csv is not available yet.")

Skipped: diabetes.csv is not available yet.


### 1.4 Handle impossible zero readings in the common Pima dataset

In the standard Pima diabetes dataset, zero values in columns such as `Glucose`, `BloodPressure`, `SkinThickness`, `Insulin`, and `BMI` are often used as placeholders for missing measurements. For those columns, zero is not treated as a genuine measurement in this practical, so it is converted to `NaN`. The median imputer in the next step will fill these missing feature values.

**Dataset caution:** Apply this zero-to-missing rule only when your file uses the common Pima dataset convention. If your supplied dataset defines zero as a valid value, do not replace it.

In [4]:
if diabetes_available:
    zero_as_missing_columns = [
        "Glucose",
        "BloodPressure",
        "SkinThickness",
        "Insulin",
        "BMI"
    ]

    # Match columns case-insensitively.
    column_lookup = {str(col).strip().lower(): col for col in X.columns}
    replaced_columns = []

    for requested_name in zero_as_missing_columns:
        actual_name = column_lookup.get(requested_name.lower())
        if actual_name is not None:
            X[actual_name] = X[actual_name].replace(0, np.nan)
            replaced_columns.append(actual_name)

    print("Columns in which zero was treated as missing:", replaced_columns)
    print("Missing values after this step:")
    display(X.isna().sum().to_frame("Missing values"))
else:
    print("Skipped: diabetes.csv is not available yet.")

Skipped: diabetes.csv is not available yet.


### 1.5 Split the dataset into training and testing sets

We use 80% of the records for training and 20% for testing. `random_state=42` makes the split reproducible. `stratify=y` keeps the class proportions approximately similar in both sets.

In [5]:
if diabetes_available:
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=42,
        stratify=y
    )

    print("Training feature records:", X_train.shape[0])
    print("Testing feature records:", X_test.shape[0])
else:
    print("Skipped: diabetes.csv is not available yet.")

Skipped: diabetes.csv is not available yet.


### 1.6 Build and train the KNN model

We use a scikit-learn `Pipeline` so the preprocessing steps are fitted on the training data only:

1. `SimpleImputer(strategy="median")` fills missing feature values using the median.
2. `StandardScaler()` scales the features.
3. `KNeighborsClassifier(n_neighbors=5)` creates the KNN classifier with K = 5.

Using a pipeline helps prevent information from the test set leaking into training-time preprocessing.

In [6]:
if diabetes_available:
    knn_model = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("knn", KNeighborsClassifier(n_neighbors=5))
    ])

    knn_model.fit(X_train, y_train)
    print("KNN model trained successfully with K = 5.")
else:
    print("Skipped: diabetes.csv is not available yet.")

Skipped: diabetes.csv is not available yet.


### 1.7 Predict the test-set values

In [7]:
if diabetes_available:
    y_pred = knn_model.predict(X_test)

    print("First 20 actual labels:")
    print(y_test.iloc[:20].to_list())

    print("\nFirst 20 predicted labels:")
    print(y_pred[:20].tolist())
else:
    print("Skipped: diabetes.csv is not available yet.")

Skipped: diabetes.csv is not available yet.


### 1.8 Calculate and display the confusion matrix

In [8]:
if diabetes_available:
    cm = confusion_matrix(y_test, y_pred, labels=[0, 1])

    print("Confusion Matrix (rows = actual; columns = predicted):")
    print(cm)

    display_labels = ["No diabetes (0)", "Diabetes (1)"]
    ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=display_labels
    ).plot(cmap="Blues", values_format="d")

    plt.title("KNN Confusion Matrix — Diabetes Dataset")
    plt.tight_layout()
    plt.show()
else:
    print("Skipped: diabetes.csv is not available yet.")

Skipped: diabetes.csv is not available yet.


**How to read the printed matrix:**

With `labels=[0, 1]`, the matrix is arranged as:

\[
\begin{bmatrix}
TN & FP \\
FN & TP
\end{bmatrix}
\]

The first row represents actual class 0 and the second row represents actual class 1. The first column represents predicted class 0 and the second column represents predicted class 1.

### 1.9 Calculate accuracy, error rate, precision, and recall

In [9]:
if diabetes_available:
    accuracy = accuracy_score(y_test, y_pred)
    error_rate = 1 - accuracy
    precision = precision_score(y_test, y_pred, pos_label=1, zero_division=0)
    recall = recall_score(y_test, y_pred, pos_label=1, zero_division=0)

    print(f"Accuracy  : {accuracy:.4f} ({accuracy * 100:.2f}%)")
    print(f"Error Rate: {error_rate:.4f} ({error_rate * 100:.2f}%)")
    print(f"Precision : {precision:.4f} ({precision * 100:.2f}%)")
    print(f"Recall    : {recall:.4f} ({recall * 100:.2f}%)")

    print("\nClassification report:")
    print(classification_report(
        y_test,
        y_pred,
        labels=[0, 1],
        target_names=["No diabetes (0)", "Diabetes (1)"],
        zero_division=0
    ))
else:
    print("Skipped: diabetes.csv is not available yet.")

Skipped: diabetes.csv is not available yet.


### 1.10 Practical 1 — Result and interpretation

After running the notebook with the actual dataset, record the values printed in the previous cell:

| Measure | Value from your run |
|---|---|
| Accuracy | __________________ |
| Error rate | __________________ |
| Precision | __________________ |
| Recall | __________________ |

**Result statement to write in the journal:**  
“The K-Nearest Neighbors classification algorithm was implemented on the diabetes dataset using K = 5. The confusion matrix was generated, and accuracy, error rate, precision, and recall were calculated on the test dataset.”

**Interpretation tip:** Do not write that a model is “good” based only on accuracy. Check precision and recall too, especially because false negatives and false positives have different meanings in a medical classification example. This model is for learning only and must not be used for medical decisions.

---

## Practical 2 — K-Means Clustering on `sales_data_sample.csv`

### Aim

To implement K-Means clustering on the sales dataset and determine a suitable number of clusters using the Elbow Method.

### Theory: What is clustering?

Clustering is an **unsupervised machine learning technique**. It groups records based on similarity without using a pre-existing target label. In a sales dataset, clustering can help reveal groups of records with similar order quantities, prices, sales values, or other numeric measures.

### What is K-Means?

K-Means divides the data into K groups. Each group is represented by its centroid (the mean position of the points in that cluster). Every record is assigned to the cluster whose centroid is nearest.

### Working of K-Means

1. Choose the number of clusters K.
2. Initialize K centroids.
3. Calculate the distance from each record to each centroid.
4. Assign each record to the closest centroid.
5. Recalculate each centroid as the mean of the records assigned to it.
6. Repeat the assignment and centroid updates until the assignments or centroids stop changing significantly.

### What is WCSS / inertia?

Within-Cluster Sum of Squares (WCSS) is the sum of squared distances between each record and the centroid of its assigned cluster. In scikit-learn, K-Means exposes this value as `inertia_`.

A lower WCSS means points are closer to their assigned centroids. However, WCSS generally decreases as K increases, so choosing the largest K is not useful by itself.

### What is the Elbow Method?

The Elbow Method runs K-Means for several values of K (for example, 1 through 10), records the WCSS for each K, and plots K against WCSS. Choose a K near the point where the curve changes from a steep decrease to a more gradual decrease. This bend is called the **elbow**.

> The elbow is a visual heuristic, not a guaranteed mathematically unique answer. If the curve has no clear bend, compare the plot with cluster interpretability and, if permitted, another measure such as the silhouette score.

### Why is scaling important?

K-Means uses distance calculations. We scale numeric features so a feature with a large unit or range does not dominate the clustering simply because of its magnitude.

### Procedure

1. Load `sales_data_sample.csv`.
2. Inspect the columns and data types.
3. Select meaningful numeric sales features, such as `QUANTITYORDERED`, `PRICEEACH`, `SALES`, and `MSRP`, if they exist.
4. Convert selected features to numeric, replace infinite values with missing values, and impute missing values using the median.
5. Scale the selected features using `StandardScaler`.
6. Calculate WCSS for K = 1 to 10 (or the maximum feasible K for a small dataset).
7. Plot the Elbow graph and identify the elbow point.
8. Set `selected_k` to the chosen value and run K-Means.
9. Display cluster labels, cluster sizes, and cluster profiles.

### 2.1 Import clustering libraries

In [10]:
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from scipy.cluster.hierarchy import linkage, dendrogram

print("Clustering libraries imported successfully.")

Clustering libraries imported successfully.


### 2.2 Load and inspect the sales dataset

In [11]:
SALES_PATH = Path("sales_data_sample.csv")
sales_available = SALES_PATH.exists()

if sales_available:
    df_sales = pd.read_csv(SALES_PATH)
    print("Sales dataset loaded successfully.")
    print("Shape (rows, columns):", df_sales.shape)
    display(df_sales.head())
    print("\nColumn names:")
    print(df_sales.columns.tolist())
    print("\nData types:")
    print(df_sales.dtypes)
else:
    print("File not found: sales_data_sample.csv")
    print("Place sales_data_sample.csv in the same folder as this notebook, then rerun this cell and the cells below.")

File not found: sales_data_sample.csv
Place sales_data_sample.csv in the same folder as this notebook, then rerun this cell and the cells below.


### 2.3 Select numeric features for clustering

The commonly distributed `sales_data_sample.csv` often contains columns such as `QUANTITYORDERED`, `PRICEEACH`, `SALES`, and `MSRP`. The cell below selects these columns when present. If fewer than two are available, it attempts a fallback using numeric-looking, non-identifier columns.

Identifiers such as order numbers, customer names, phone numbers, and postal codes are not useful distance features for this exercise. Categorical text columns also cannot be passed directly to K-Means.

In [12]:
if sales_available:
    # Match common feature names case-insensitively.
    original_by_upper = {str(col).strip().upper(): col for col in df_sales.columns}
    preferred_feature_names = ["QUANTITYORDERED", "PRICEEACH", "SALES", "MSRP"]
    selected_sales_features = [
        original_by_upper[name]
        for name in preferred_feature_names
        if name in original_by_upper
    ]

    # Fallback: identify numeric-looking columns if the standard fields are unavailable.
    if len(selected_sales_features) < 2:
        identifier_terms = [
            "ORDERNUMBER", "ORDERLINENUMBER", "CUSTOMER", "PHONE",
            "POSTAL", "ZIP", "ADDRESS", "YEAR_ID", "MONTH_ID", "QTR_ID"
        ]
        fallback_features = []

        for column in df_sales.columns:
            upper_name = str(column).strip().upper()
            if any(term in upper_name for term in identifier_terms):
                continue
            numeric_candidate = pd.to_numeric(df_sales[column], errors="coerce")
            non_missing_fraction = numeric_candidate.notna().mean()
            if non_missing_fraction >= 0.70 and numeric_candidate.nunique(dropna=True) > 1:
                fallback_features.append(column)

        for column in fallback_features:
            if column not in selected_sales_features:
                selected_sales_features.append(column)

    selected_sales_features = selected_sales_features[:4]

    if len(selected_sales_features) < 2:
        raise ValueError(
            "Could not identify at least two usable numeric sales features. "
            "Inspect df_sales.columns and manually choose at least two numeric columns."
        )

    sales_feature_data = df_sales[selected_sales_features].apply(pd.to_numeric, errors="coerce")
    sales_feature_data = sales_feature_data.replace([np.inf, -np.inf], np.nan)

    # Remove any selected feature that is completely missing after conversion.
    sales_feature_data = sales_feature_data.dropna(axis=1, how="all")
    selected_sales_features = sales_feature_data.columns.tolist()

    if len(selected_sales_features) < 2:
        raise ValueError("Fewer than two usable numeric columns remain after cleaning.")

    print("Selected clustering features:", selected_sales_features)
    print("Feature data shape:", sales_feature_data.shape)
    display(sales_feature_data.head())
else:
    print("Skipped: sales_data_sample.csv is not available yet.")

Skipped: sales_data_sample.csv is not available yet.


### 2.4 Impute missing values and scale the features

The median imputer fills missing values in each selected feature. The scaler then standardizes the features. We keep the original row index so that cluster labels can be joined back to the matching sales records.

In [13]:
if sales_available:
    sales_imputer = SimpleImputer(strategy="median")
    sales_scaler = StandardScaler()

    sales_imputed_array = sales_imputer.fit_transform(sales_feature_data)
    sales_scaled_array = sales_scaler.fit_transform(sales_imputed_array)

    sales_imputed = pd.DataFrame(
        sales_imputed_array,
        columns=selected_sales_features,
        index=sales_feature_data.index
    )
    X_sales_scaled = pd.DataFrame(
        sales_scaled_array,
        columns=selected_sales_features,
        index=sales_feature_data.index
    )

    print("Missing values after median imputation:")
    display(sales_imputed.isna().sum().to_frame("Missing values"))
    print("Scaled feature preview:")
    display(X_sales_scaled.head())
else:
    print("Skipped: sales_data_sample.csv is not available yet.")

Skipped: sales_data_sample.csv is not available yet.


### 2.5 Calculate WCSS for different K values

We fit a K-Means model for each K from 1 to 10, then save `inertia_` (WCSS) for the Elbow graph. `n_init=10` runs the algorithm from multiple initial centroid settings and keeps the best result. `random_state=42` makes the result reproducible.

In [14]:
if sales_available:
    if len(X_sales_scaled) < 2:
        raise ValueError("At least two records are needed for clustering.")

    max_k = min(10, len(X_sales_scaled) - 1)
    k_values = list(range(1, max_k + 1))
    wcss = []

    for k in k_values:
        kmeans_test = KMeans(
            n_clusters=k,
            random_state=42,
            n_init=10
        )
        kmeans_test.fit(X_sales_scaled)
        wcss.append(kmeans_test.inertia_)

    print("K values tested:", k_values)
    print("WCSS values:", [round(value, 2) for value in wcss])
else:
    print("Skipped: sales_data_sample.csv is not available yet.")

Skipped: sales_data_sample.csv is not available yet.


### 2.6 Plot the Elbow graph and choose K

In [15]:
if sales_available:
    plt.figure(figsize=(8, 5))
    plt.plot(k_values, wcss, marker="o")
    plt.title("Elbow Method for K-Means Clustering")
    plt.xlabel("Number of clusters (K)")
    plt.ylabel("WCSS / Inertia")
    plt.xticks(k_values)
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

    print("Inspect the graph and identify the point where the WCSS decrease begins to slow down.")
else:
    print("Skipped: sales_data_sample.csv is not available yet.")

Skipped: sales_data_sample.csv is not available yet.


**How to choose the elbow:** Look for the bend in the curve. Before the bend, increasing K reduces WCSS substantially. After the bend, adding more clusters produces smaller improvements. The correct K depends on the graph from your file; do not copy a fixed K from another student's output.

### 2.7 Fit the final K-Means model

In [16]:
if sales_available:
    # IMPORTANT: Change this after inspecting your Elbow graph.
    selected_k = 3

    if selected_k < 2 or selected_k > len(X_sales_scaled):
        raise ValueError("selected_k must be at least 2 and no greater than the number of records.")

    kmeans_model = KMeans(
        n_clusters=selected_k,
        random_state=42,
        n_init=10
    )
    kmeans_labels = kmeans_model.fit_predict(X_sales_scaled)

    kmeans_results = df_sales.loc[X_sales_scaled.index].copy()
    kmeans_results["KMeans_Cluster"] = kmeans_labels

    print(f"K-Means completed with K = {selected_k}.")
    print("First 10 cluster labels:", kmeans_labels[:10].tolist())
    print("Cluster sizes:")
    display(kmeans_results["KMeans_Cluster"].value_counts().sort_index().to_frame("Number of records"))
else:
    print("Skipped: sales_data_sample.csv is not available yet.")

Skipped: sales_data_sample.csv is not available yet.


### 2.8 Examine the cluster profiles

In [17]:
if sales_available:
    cluster_profile = kmeans_results.groupby("KMeans_Cluster")[selected_sales_features].mean().round(2)
    print("Average feature values for each K-Means cluster:")
    display(cluster_profile)

    print("Interpretation guide: compare the mean sales, quantity, price, or other selected features across clusters.")
    print("Cluster numbers are labels only; Cluster 2 is not automatically better or larger than Cluster 1.")
else:
    print("Skipped: sales_data_sample.csv is not available yet.")

Skipped: sales_data_sample.csv is not available yet.


### 2.9 Visualize K-Means clusters

The following chart uses the first two selected features for a simple two-dimensional view. Cluster assignments were calculated using all selected features, not only the two shown in the plot.

In [18]:
if sales_available:
    plot_x = selected_sales_features[0]
    plot_y = selected_sales_features[1]

    plt.figure(figsize=(8, 5))
    scatter = plt.scatter(
        sales_imputed[plot_x],
        sales_imputed[plot_y],
        c=kmeans_labels,
        cmap="viridis",
        alpha=0.75
    )
    plt.title("K-Means Clusters — Sales Dataset")
    plt.xlabel(plot_x)
    plt.ylabel(plot_y)
    plt.colorbar(scatter, label="Cluster label")
    plt.tight_layout()
    plt.show()
else:
    print("Skipped: sales_data_sample.csv is not available yet.")

Skipped: sales_data_sample.csv is not available yet.


### 2.10 Practical 2 — Result and interpretation

**Result statement to write in the journal:**  
“The K-Means clustering algorithm was implemented on the sales dataset. The Elbow Method was used to inspect WCSS values for different K values. The selected K was used to assign each record to a cluster, and cluster sizes and average feature values were examined.”

Fill in after running:

- Selected features: __________________________________________
- Elbow point / selected K: ____________________________________
- Number of clusters produced: ________________________________
- Brief description of the clusters: ___________________________

**Interpretation tip:** Give each cluster a business description only after checking its average feature values. For example, describe a cluster as “higher average sales” only if the profile table supports that statement.

---

## Practical 3 — Agglomerative Hierarchical Clustering on `sales_data_sample.csv`

### Aim

To implement Agglomerative Hierarchical Clustering on the scaled numeric sales data and interpret the dendrogram to understand how clusters are merged.

### Theory: What is Hierarchical Clustering?

Hierarchical clustering is an unsupervised learning technique that builds a hierarchy of nested clusters. Unlike K-Means, it does not begin by assigning every record to one of K centroids.

There are two main approaches:

1. **Agglomerative (bottom-up):** Start with every record as its own cluster and repeatedly merge the closest clusters.
2. **Divisive (top-down):** Start with all records in one cluster and repeatedly split them into smaller clusters.

This practical uses **Agglomerative Clustering**.

### What is a dendrogram?

A dendrogram is a tree-shaped diagram that shows the order and distance at which clusters are merged. The vertical height of a merge represents the distance (or linkage cost) at which two groups are joined. A large vertical gap can suggest a useful place to cut the tree into separate clusters.

### Linkage method used: Ward

Ward linkage chooses merges that minimize the increase in within-cluster variance. It is commonly used with Euclidean distance and scaled numeric data. This practical uses `linkage="ward"`.

### Procedure

1. Reuse the cleaned and scaled sales features from Practical 2.
2. If the dataset is extremely large, take a reproducible sample for hierarchical clustering because hierarchical methods can require substantial time and memory.
3. Create a linkage matrix using SciPy's `linkage()` function with Ward's method.
4. Plot a dendrogram.
5. Inspect the dendrogram and choose a reasonable number of clusters.
6. Fit `AgglomerativeClustering` with that number of clusters.
7. Display the cluster labels and profile each cluster.

> **Important distinction:** The Elbow Method is normally associated with K-Means. For hierarchical clustering, the dendrogram is the standard visual tool for deciding where to cut the hierarchy. You may start with the K selected from the Elbow graph for comparison, but inspect the dendrogram independently.

### 3.1 Prepare data for hierarchical clustering

In [19]:
if sales_available:
    # Hierarchical clustering can become expensive as the number of records grows.
    # Use all rows up to this limit; otherwise take a reproducible sample.
    MAX_HIERARCHICAL_ROWS = 3000

    if len(X_sales_scaled) > MAX_HIERARCHICAL_ROWS:
        X_hierarchical = X_sales_scaled.sample(
            n=MAX_HIERARCHICAL_ROWS,
            random_state=42
        )
        print(f"Using a reproducible sample of {MAX_HIERARCHICAL_ROWS} rows for hierarchical clustering.")
    else:
        X_hierarchical = X_sales_scaled.copy()
        print(f"Using all {len(X_hierarchical)} rows for hierarchical clustering.")

    print("Hierarchical input shape:", X_hierarchical.shape)
else:
    print("Skipped: sales_data_sample.csv is not available yet.")

Skipped: sales_data_sample.csv is not available yet.


### 3.2 Calculate the linkage matrix

In [20]:
if sales_available:
    linkage_matrix = linkage(
        X_hierarchical.to_numpy(),
        method="ward"
    )

    print("Linkage matrix created.")
    print("Linkage matrix shape:", linkage_matrix.shape)
else:
    print("Skipped: sales_data_sample.csv is not available yet.")

Skipped: sales_data_sample.csv is not available yet.


### 3.3 Plot the dendrogram

In [21]:
if sales_available:
    plt.figure(figsize=(12, 6))
    dendrogram(
        linkage_matrix,
        truncate_mode="lastp",
        p=30,
        show_leaf_counts=True
    )
    plt.title("Hierarchical Clustering Dendrogram (Ward Linkage)")
    plt.xlabel("Merged data points / clusters")
    plt.ylabel("Ward linkage distance")
    plt.tight_layout()
    plt.show()

    print("Inspect large vertical gaps and decide where a horizontal cut could separate groups.")
else:
    print("Skipped: sales_data_sample.csv is not available yet.")

Skipped: sales_data_sample.csv is not available yet.


**How to interpret the dendrogram:** Imagine drawing a horizontal line across the tree. The number of vertical branches that the line crosses is the approximate number of clusters produced by that cut. A cut below a large merge height can keep groups separate; a cut above it merges them. The choice should also produce clusters that are useful and interpretable.

### 3.4 Fit Agglomerative Clustering

In [22]:
if sales_available:
    # Update this after inspecting the dendrogram.
    # It is initialized to the K-Means choice only to make comparison easier.
    hierarchical_k = selected_k if "selected_k" in globals() else 3

    if hierarchical_k < 2 or hierarchical_k > len(X_hierarchical):
        raise ValueError("hierarchical_k must be at least 2 and no greater than the hierarchical sample size.")

    hierarchical_model = AgglomerativeClustering(
        n_clusters=hierarchical_k,
        linkage="ward"
    )
    hierarchical_labels = hierarchical_model.fit_predict(X_hierarchical)

    hierarchical_results = df_sales.loc[X_hierarchical.index].copy()
    hierarchical_results["Hierarchical_Cluster"] = hierarchical_labels

    print(f"Agglomerative clustering completed with {hierarchical_k} clusters.")
    print("First 10 hierarchical labels:", hierarchical_labels[:10].tolist())
    print("Cluster sizes:")
    display(
        hierarchical_results["Hierarchical_Cluster"]
        .value_counts()
        .sort_index()
        .to_frame("Number of records")
    )
else:
    print("Skipped: sales_data_sample.csv is not available yet.")

Skipped: sales_data_sample.csv is not available yet.


### 3.5 Examine hierarchical cluster profiles

In [23]:
if sales_available:
    hierarchical_profile = (
        hierarchical_results
        .groupby("Hierarchical_Cluster")[selected_sales_features]
        .mean()
        .round(2)
    )
    print("Average feature values for each hierarchical cluster:")
    display(hierarchical_profile)
else:
    print("Skipped: sales_data_sample.csv is not available yet.")

Skipped: sales_data_sample.csv is not available yet.


### 3.6 Visualize hierarchical clusters

In [24]:
if sales_available:
    plot_x = selected_sales_features[0]
    plot_y = selected_sales_features[1]

    plt.figure(figsize=(8, 5))
    scatter = plt.scatter(
        sales_imputed.loc[X_hierarchical.index, plot_x],
        sales_imputed.loc[X_hierarchical.index, plot_y],
        c=hierarchical_labels,
        cmap="plasma",
        alpha=0.75
    )
    plt.title("Agglomerative Hierarchical Clusters — Sales Dataset")
    plt.xlabel(plot_x)
    plt.ylabel(plot_y)
    plt.colorbar(scatter, label="Cluster label")
    plt.tight_layout()
    plt.show()
else:
    print("Skipped: sales_data_sample.csv is not available yet.")

Skipped: sales_data_sample.csv is not available yet.


### 3.7 Practical 3 — Result and interpretation

**Result statement to write in the journal:**  
“Agglomerative Hierarchical Clustering was implemented on the scaled numeric sales data using Ward linkage. A dendrogram was plotted to examine the merging structure. The selected number of clusters was used to fit the model, and cluster labels and cluster profiles were obtained.”

Fill in after running:

- Number of records used for hierarchical clustering: __________________
- Linkage method: Ward
- Number of clusters selected from dendrogram: _______________________
- Brief description of the clusters: _________________________________

---

## Comparison: KNN, K-Means, and Hierarchical Clustering

| Point | KNN | K-Means | Agglomerative Hierarchical |
|---|---|---|---|
| Learning type | Supervised | Unsupervised | Unsupervised |
| Main purpose | Classification / regression | Clustering | Clustering |
| Needs labelled target? | Yes, for supervised training | No | No |
| Main idea | Vote among nearest training records | Assign records to nearest centroid | Repeatedly merge closest groups |
| Number of groups | Not applicable; predicts target class | K must be selected | Choose a cut / number of clusters |
| Common visual / evaluation | Confusion matrix and metrics | Elbow graph | Dendrogram |

### Common practical mistakes to avoid

1. Do not use the target column `Outcome` as an input feature in the KNN model.
2. Do not fit the scaler on the full diabetes data before the train/test split. The pipeline avoids this leakage.
3. Do not skip scaling for KNN or K-Means when feature units/ranges differ.
4. Do not use text columns directly in K-Means distance calculations.
5. Do not assume K = 3 without inspecting the Elbow graph and dendrogram.
6. Do not interpret cluster numbers as rankings. Cluster IDs are arbitrary labels.
7. Do not report example or guessed metric values. Copy the values produced by your own dataset run.

## Viva voce questions and answers

### 1. What is machine learning?
Machine learning is a field of computing in which algorithms learn patterns from data to make predictions or discover structure.

### 2. What is supervised learning?
Supervised learning trains a model using labelled examples, where the correct target is known. KNN classification is a supervised method.

### 3. What is unsupervised learning?
Unsupervised learning finds patterns or groups in data without a supplied target label. K-Means and hierarchical clustering are examples.

### 4. What is KNN?
KNN is a supervised algorithm that predicts a record's class from the majority class among its K nearest training records.

### 5. What does K mean in KNN?
K is the number of nearest neighbours used to make a prediction.

### 6. Why is scaling important in KNN?
KNN calculates distances. Scaling stops large-range features from dominating smaller-range features.

### 7. What is a confusion matrix?
It is a table comparing actual and predicted classes, showing true positives, true negatives, false positives, and false negatives.

### 8. What is accuracy?
Accuracy is the proportion of all predictions that are correct.

### 9. What is precision?
Precision is the proportion of predicted positive records that are truly positive.

### 10. What is recall?
Recall is the proportion of actual positive records correctly identified by the model.

### 11. What is the error rate?
The error rate is the proportion of predictions that are incorrect. It equals 1 minus accuracy.

### 12. What is K-Means?
K-Means is an unsupervised algorithm that groups records into K clusters by assigning them to the nearest centroid and updating the centroids.

### 13. What is a centroid?
A centroid is the mean position of the records assigned to a cluster in feature space.

### 14. What is WCSS?
WCSS is the sum of squared distances from each record to the centroid of its assigned cluster. In scikit-learn it is available as `inertia_`.

### 15. What is the Elbow Method?
It plots WCSS for different K values and helps choose a K near the point where the improvement begins to slow down.

### 16. What is hierarchical clustering?
It is an unsupervised method that builds a hierarchy of clusters by repeatedly merging or splitting groups.

### 17. What is Agglomerative clustering?
It is the bottom-up form of hierarchical clustering: each record begins in its own cluster, and clusters are repeatedly merged.

### 18. What is a dendrogram?
A dendrogram is a tree diagram showing the order and distance at which clusters are merged.

### 19. What is Ward linkage?
Ward linkage chooses the merge that produces the smallest increase in within-cluster variance.

### 20. What is the difference between KNN and K-Means?
KNN is supervised and predicts a label using nearby labelled examples. K-Means is unsupervised and groups unlabelled records around centroids.

### 21. Why do we use `random_state=42`?
It makes random operations such as splitting or initialization reproducible. The number 42 is a convention; another fixed number can also be used.

### 22. Why do we use median imputation?
It fills missing numeric values with the median of a feature. The median is less affected by extreme values than the mean.

---

## Final checklist before the practical exam

- [ ] I can explain the difference between supervised and unsupervised learning.
- [ ] I can explain KNN and the meaning of K.
- [ ] I can separate X and y in the diabetes dataset.
- [ ] I can explain why the data is split into training and testing sets.
- [ ] I can identify TN, FP, FN, and TP in the confusion matrix.
- [ ] I can write the formulas for accuracy, error rate, precision, and recall.
- [ ] I can explain why feature scaling is needed.
- [ ] I can explain K-Means, centroids, and WCSS.
- [ ] I can identify the elbow point from the graph.
- [ ] I can explain Agglomerative clustering and a dendrogram.
- [ ] I can describe the clusters using their average feature values.

## Overall conclusion

The first practical demonstrates supervised classification using KNN on diabetes records and evaluates its predictions using standard classification metrics. The second and third practicals demonstrate unsupervised grouping of sales records using K-Means and Agglomerative Hierarchical Clustering. The Elbow graph supports the selection of K for K-Means, while the dendrogram helps interpret the hierarchical merging structure.